This notebook estimates the taxonomic depth based on WordNet.

In [1]:
import pandas as pd
import nltk
from nltk.corpus import wordnet as wn
import os

Print version numbers for reproducibility

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-09-25T10:06:39.864912+10:00

Python implementation: CPython
Python version       : 3.11.5
IPython version      : 8.12.3

Compiler    : MSC v.1936 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 154 Stepping 4, GenuineIntel
CPU cores   : 12
Architecture: 64bit

pandas: 2.2.3
nltk  : 3.8.1



In [3]:
# Synset ID
synset_id = "a00023854"

# Extract POS (first character) and offset (the rest)
pos = synset_id[0]
offset = int(synset_id[1:])

# Get synset
synset = wn.synset_from_pos_and_offset(pos, offset)

# Compute taxonomic depth
depth = synset.min_depth()

print(f"Synset: {synset}")
print(f"Definition: {synset.definition()}")
print(f"Taxonomic Depth: {depth}")

Synset: Synset('faulty.s.02')
Definition: characterized by errors; not agreeing with a model or not following established rules; ; ; the wrong side of the road"
Taxonomic Depth: 0


In [4]:
project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

file_path = os.path.join(project_folder, "data", "synonym_mapping.csv")
df = pd.read_csv(file_path, encoding='utf-8')

Extract information about taxonomic depth.

In [5]:
def get_taxonomic_depth(synset_id):
    try:
        pos = synset_id[0]
        offset = int(synset_id[1:])
        synset = wn.synset_from_pos_and_offset(pos, offset)
        return synset.min_depth()
    except:
        return None  

df['taxonomic_depth'] = df['final_synset'].apply(get_taxonomic_depth)

print(df.head())

   concepticon_id        name  \
0               2        dust   
1               3       brave   
2               3       brave   
3               6  earthquake   
4               6  earthquake   

                                         description final_synset     synonym  \
0  Any kind of solid material divided in particle...    n14839846        dust   
1                Having or characterized by courage.    a00262792       brave   
2                Having or characterized by courage.    a00262792  courageous   
3  The violent shaking of the ground produced by ...    n07428954  earthquake   
4  The violent shaking of the ground produced by ...    n07428954       quake   

  wordnet_id   count                                         definition  \
0  n14839846  1353.0  fine powdery material such as dry earth or pol...   
1  a00262792   489.0  possessing or displaying courage; able to face...   
2  a00262792    87.0  possessing or displaying courage; able to face...   
3  n07428954  

In [6]:
taxon_depth_file = os.path.join(project_folder, "data", "taxon_depth.tsv")
df.to_csv(taxon_depth_file, sep='\t', index=False, encoding='utf-8')